# LeetCode 2034 — Stock Price Fluctuation bằng Python

Link đề: https://leetcode.com/problems/stock-price-fluctuation/ · Độ khó: **Medium** · Chủ đề: Hash Table, Design, Heap (Priority Queue), Data Stream, Ordered Set

Skeleton LeetCode:

```python
class StockPrice:

    def __init__(self):
        ...

    def update(self, timestamp: int, price: int) -> None:
        ...

    def current(self) -> int:
        ...

    def maximum(self) -> int:
        ...

    def minimum(self) -> int:
        ...
```

## 1. Mục tiêu học

1. Thiết kế lớp với nhiều thao tác và chọn **cấu trúc dữ liệu phù hợp cho từng thao tác**: `dict` (giá theo thời điểm), biến `latest` (thời điểm lớn nhất), heap / tập có thứ tự (max, min).
2. Hiểu và cài **xóa lười** (lazy deletion) trên heap: không xóa phần tử cũ ngay, chỉ bỏ qua khi nó nổi lên đỉnh mà đã "hết hạn".
3. Dùng **danh sách có thứ tự** (`SortedList` trên LeetCode, hoặc tự cài bằng `bisect`) để hỗ trợ thêm/xóa và lấy min/max.
4. Viết **bộ chạy chuỗi lệnh** kiểu LeetCode và đối chiếu với cài đặt brute force trên chuỗi lệnh ngẫu nhiên seed cố định.
5. Tránh các bẫy: quên xóa giá cũ khi **sửa** bản ghi, nhầm "mới nhất" là "vừa cập nhật", kiểm tra hạn sai trong xóa lười.

## 2. Đề bài & ví dụ

Có một luồng bản ghi `(timestamp, price)` của một cổ phiếu. Bản ghi **không đến theo thứ tự** thời gian, và có thể **sai**: một bản ghi sau cùng `timestamp` sẽ **sửa** giá của bản ghi trước.

Cài lớp `StockPrice`:
- `update(timestamp, price)`: đặt (hoặc sửa) giá tại `timestamp`.
- `current()`: giá tại **thời điểm lớn nhất** đã ghi nhận.
- `maximum()`: giá **lớn nhất** trong các bản ghi hiện tại (sau khi sửa).
- `minimum()`: giá **nhỏ nhất** trong các bản ghi hiện tại.

**Ví dụ:**

```
["StockPrice","update","update","current","maximum","update","maximum","update","minimum"]
[[],[1,10],[2,5],[],[],[1,3],[],[4,2],[]]
→ [null,null,null,5,10,null,5,null,2]
```

Sau `update(1,10)`, `update(2,5)`: `current = 5` (thời điểm 2), `maximum = 10`. Sau `update(1,3)` (sửa thời điểm 1 từ 10 thành 3): `maximum = 5`. Sau `update(4,2)`: `minimum = 2`.

**Ràng buộc:** `1 ≤ timestamp, price ≤ 10⁹`; tổng cộng tối đa `10⁵` lời gọi; `current/maximum/minimum` chỉ được gọi sau ít nhất một `update`.

In [1]:
import random
import time
import heapq
from bisect import bisect_left, insort
from collections import Counter

ops_example = ["StockPrice", "update", "update", "current", "maximum", "update", "maximum", "update", "minimum"]
args_example = [[], [1, 10], [2, 5], [], [], [1, 3], [], [4, 2], []]
expected_example = [None, None, None, 5, 10, None, 5, None, 2]

def run_ops(cls, ops, args):
    """Bộ chạy chuỗi lệnh kiểu LeetCode: lệnh đầu là khởi tạo."""
    obj = cls(*args[0]); out = [None]
    for op, a in zip(ops[1:], args[1:]):
        out.append(getattr(obj, op)(*a))
    return out
print(list(zip(ops_example, args_example)))

[('StockPrice', []), ('update', [1, 10]), ('update', [2, 5]), ('current', []), ('maximum', []), ('update', [1, 3]), ('maximum', []), ('update', [4, 2]), ('minimum', [])]


## 3. Kiến thức nền

### 3.1 `dict` — giá theo thời điểm, ghi đè khi sửa

`d[ts] = price` vừa thêm mới vừa sửa. Kiểm tra đã có chưa: `ts in d`. Đây là "nguồn sự thật" (source of truth) của mọi cấu trúc khác.

In [2]:
d = {}
d[1] = 10; d[2] = 5
print(d, "| đã có ts 1?", 1 in d)
d[1] = 3
print("sau khi sửa:", d)

{1: 10, 2: 5} | đã có ts 1? True
sau khi sửa: {1: 3, 2: 5}


### 3.2 `heapq` — min-heap; max-heap bằng đổi dấu

`heapq.heappush(h, x)`, `heapq.heappop(h)`, `h[0]` là nhỏ nhất. Max-heap: đẩy `-x`. Heap **không hỗ trợ xóa phần tử bất kỳ** hiệu quả — đó là lý do cần xóa lười.

In [3]:
mn, mx = [], []
for p in [10, 5, 3]:
    heapq.heappush(mn, p); heapq.heappush(mx, -p)
print("nhỏ nhất:", mn[0], "| lớn nhất:", -mx[0])

nhỏ nhất: 3 | lớn nhất: 10


### 3.3 Xóa lười (lazy deletion)

Đẩy vào heap cặp `(price, ts)`. Khi cần đỉnh: nếu `d[ts] != price` thì bản ghi đó đã bị sửa ⇒ **bỏ đi** (pop) và xem phần tử tiếp theo. Mỗi phần tử bị pop tối đa một lần ⇒ tổng chi phí khấu hao `O(log n)` mỗi thao tác.

In [4]:
d = {1: 10, 2: 5}
h = [(-10, 1), (-5, 2)]; heapq.heapify(h)
d[1] = 3; heapq.heappush(h, (-3, 1))          # sửa ts 1: không xóa (-10, 1) ngay
while d[h[0][1]] != -h[0][0]:
    print("bỏ bản ghi hết hạn:", h[0]); heapq.heappop(h)
print("max hiện tại:", -h[0][0])

bỏ bản ghi hết hạn: (-10, 1)
max hiện tại: 5


### 3.4 Danh sách có thứ tự bằng `bisect`

LeetCode cài sẵn thư viện `sortedcontainers` (`SortedList`: thêm/xóa `O(log n)` khấu hao). Môi trường chạy notebook này **không có** thư viện đó, nên ta tự cài một lớp nhỏ cùng API bằng `bisect.insort` (thêm) và `bisect_left` + `pop` (xóa) — đúng đắn nhưng thêm/xóa `O(n)` do dịch phần tử (rất nhanh trong C).

In [5]:
class MiniSortedList:
    """Thay thế tối giản cho sortedcontainers.SortedList: add, remove, [i], len."""
    def __init__(self):
        self.a = []
    def add(self, x):
        insort(self.a, x)
    def remove(self, x):
        i = bisect_left(self.a, x)
        if i == len(self.a) or self.a[i] != x:
            raise ValueError("không có phần tử")
        self.a.pop(i)
    def __getitem__(self, i):
        return self.a[i]
    def __len__(self):
        return len(self.a)

sl = MiniSortedList()
for p in [10, 5, 3, 5]: sl.add(p)
sl.remove(5)
print(sl.a, "min =", sl[0], "max =", sl[-1])

[3, 5, 10] min = 3 max = 10


## 4. Ý tưởng nền tảng / trực giác

Mỗi thao tác cần một "câu hỏi" khác nhau:

| Thao tác | Cần biết | Cấu trúc tự nhiên |
|---|---|---|
| `update` | ghi/sửa giá tại `ts` | `dict` |
| `current` | giá tại `ts` lớn nhất | biến `latest = max(latest, ts)` + `dict` |
| `maximum` | max của mọi giá **hiện tại** | max-heap (xóa lười) / SortedList |
| `minimum` | min của mọi giá hiện tại | min-heap (xóa lười) / SortedList |

Điểm khó duy nhất: khi **sửa** giá, giá cũ phải "biến mất" khỏi max/min. Trong ví dụ: sau khi sửa `ts = 1` từ 10 thành 3, `maximum` phải là 5 chứ không phải 10.

In [6]:
d = {}; latest = 0
for ts, p in [(1, 10), (2, 5), (1, 3), (4, 2)]:
    d[ts] = p; latest = max(latest, ts)
    print(f"update({ts},{p}) -> d={d}, latest={latest}, current={d[latest]}, "
          f"max={max(d.values())}, min={min(d.values())}")

update(1,10) -> d={1: 10}, latest=1, current=10, max=10, min=10
update(2,5) -> d={1: 10, 2: 5}, latest=2, current=5, max=10, min=5
update(1,3) -> d={1: 3, 2: 5}, latest=2, current=5, max=5, min=3
update(4,2) -> d={1: 3, 2: 5, 4: 2}, latest=4, current=2, max=5, min=2


## 5. Cách 1 — Brute Force / baseline: `dict` + duyệt toàn bộ khi hỏi max/min

**Ý tưởng:** chỉ giữ `dict` và `latest`. `maximum()`/`minimum()` dùng `max(d.values())` / `min(d.values())`.

In [7]:
class StockPriceBrute:
    def __init__(self):
        self.d = {}
        self.latest = 0
    def update(self, timestamp: int, price: int) -> None:
        self.d[timestamp] = price
        self.latest = max(self.latest, timestamp)
    def current(self) -> int:
        return self.d[self.latest]
    def maximum(self) -> int:
        return max(self.d.values())
    def minimum(self) -> int:
        return min(self.d.values())

print(run_ops(StockPriceBrute, ops_example, args_example), "| kỳ vọng", expected_example)

[None, None, None, 5, 10, None, 5, None, 2] | kỳ vọng [None, None, None, 5, 10, None, 5, None, 2]


**Độ phức tạp:** `update`, `current` `O(1)`; `maximum`, `minimum` `O(k)` với `k` = số thời điểm khác nhau. Worst case `10⁵` lời gọi, nửa là `update` (k tới `5·10⁴`), nửa là `maximum` ⇒ `2,5·10⁹` bước ⇒ TLE.

**Nút thắt:** ta tính lại max/min từ đầu mỗi lần hỏi, trong khi mỗi `update` chỉ thay đổi **một** giá.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** cần một cấu trúc cho biết max/min nhanh **và** chịu được việc một giá cũ bị thay thế.

- **Heap + xóa lười:** khi sửa, không xóa giá cũ; đẩy `(price, ts)` mới. Khi hỏi max: **nếu** đỉnh `(p, ts)` có `d[ts] != p` **thì** nó đã lỗi thời ⇒ pop và kiểm tiếp. **Invariant:** mọi bản ghi hiện tại `(d[ts], ts)` luôn có mặt trong heap; mọi phần tử khác trong heap là rác sẽ bị loại khi lên đỉnh.
- **Tập có thứ tự:** khi sửa, **xóa đúng** giá cũ `d[ts]` (một bản) rồi thêm giá mới. Max = phần tử cuối, min = phần tử đầu.

**Quy tắc "nếu… thì…" cho `update`:**
- **Nếu** `ts` đã có **thì** (SortedList) xóa `d[ts]` cũ; (heap) không làm gì thêm.
- Ghi `d[ts] = price`, thêm `price` vào cấu trúc, `latest = max(latest, ts)`.

Chú ý: `current` dùng `latest` (thời điểm **lớn nhất**), không phải thời điểm của lời gọi `update` gần nhất.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Hai heap + xóa lười (bản nộp chính, không cần thư viện ngoài)

**Giải thích từng đoạn:**
1. `self.d`, `self.latest`, `self.mx` (max-heap dạng `(-price, ts)`), `self.mn` (min-heap `(price, ts)`).
2. `update`: ghi `d`, cập nhật `latest`, đẩy vào **cả hai** heap.
3. `maximum`: trong khi `d[ts] != -mx[0][0]` thì pop; trả `-mx[0][0]`. `minimum` tương tự.

In [8]:
class StockPriceHeap:
    def __init__(self):
        self.d = {}
        self.latest = 0
        self.mx = []          # (-price, ts)
        self.mn = []          # (price, ts)

    def update(self, timestamp: int, price: int) -> None:
        self.d[timestamp] = price
        if timestamp > self.latest:
            self.latest = timestamp
        heapq.heappush(self.mx, (-price, timestamp))
        heapq.heappush(self.mn, (price, timestamp))

    def current(self) -> int:
        return self.d[self.latest]

    def maximum(self) -> int:
        while self.d[self.mx[0][1]] != -self.mx[0][0]:   # bản ghi lỗi thời
            heapq.heappop(self.mx)
        return -self.mx[0][0]

    def minimum(self) -> int:
        while self.d[self.mn[0][1]] != self.mn[0][0]:
            heapq.heappop(self.mn)
        return self.mn[0][0]

print(run_ops(StockPriceHeap, ops_example, args_example))

[None, None, None, 5, 10, None, 5, None, 2]


**Độ phức tạp Cách 2:** `update` `O(log m)` (`m` = số lần update), `current` `O(1)`, `maximum`/`minimum` `O(log m)` **khấu hao** (mỗi phần tử bị pop tối đa một lần trong mỗi heap). Space `O(m)` (heap giữ cả rác).

### 7.2 Cách 3 — Danh sách có thứ tự (SortedList / MiniSortedList)

**Giải thích:** `update`: nếu `ts` đã có thì `remove(d[ts])`; rồi `add(price)`. `maximum` = `sl[-1]`, `minimum` = `sl[0]`. Không có rác, bộ nhớ đúng `O(k)`.

Trên LeetCode viết `from sortedcontainers import SortedList` và dùng `SortedList()`; ở đây chạy với `MiniSortedList` cùng API.

In [9]:
class StockPriceSorted:
    def __init__(self, sorted_list_cls=MiniSortedList):
        self.d = {}
        self.latest = 0
        self.sl = sorted_list_cls()

    def update(self, timestamp: int, price: int) -> None:
        if timestamp in self.d:
            self.sl.remove(self.d[timestamp])       # xóa đúng MỘT bản giá cũ
        self.d[timestamp] = price
        self.sl.add(price)
        self.latest = max(self.latest, timestamp)

    def current(self) -> int:
        return self.d[self.latest]

    def maximum(self) -> int:
        return self.sl[-1]

    def minimum(self) -> int:
        return self.sl[0]

print(run_ops(StockPriceSorted, ops_example, args_example))

[None, None, None, 5, 10, None, 5, None, 2]


**Độ phức tạp Cách 3:** với `SortedList` thật: `update` `O(log k)`, `maximum`/`minimum` `O(1)`. Với `MiniSortedList`: `update` `O(k)` (dịch mảng, hằng số nhỏ), truy vấn `O(1)`. Space `O(k)`.

### 7.3 Cách 4 — Heap + `Counter` "số bản cần xóa" (xóa lười theo giá)

Một kiểu xóa lười khác, hữu ích khi heap chỉ chứa **giá** (không có `ts`): giữ `to_del[price]` = số bản của giá này đã lỗi thời. Khi đỉnh có `to_del[top] > 0` thì pop và giảm bộ đếm.

In [10]:
class StockPriceCounterLazy:
    def __init__(self):
        self.d = {}; self.latest = 0
        self.mx, self.mn = [], []
        self.del_mx, self.del_mn = Counter(), Counter()

    def update(self, timestamp: int, price: int) -> None:
        if timestamp in self.d:
            old = self.d[timestamp]
            self.del_mx[old] += 1; self.del_mn[old] += 1    # đánh dấu một bản giá cũ cần xóa
        self.d[timestamp] = price
        self.latest = max(self.latest, timestamp)
        heapq.heappush(self.mx, -price); heapq.heappush(self.mn, price)

    def current(self) -> int:
        return self.d[self.latest]

    def maximum(self) -> int:
        while self.del_mx[-self.mx[0]]:
            self.del_mx[-self.mx[0]] -= 1; heapq.heappop(self.mx)
        return -self.mx[0]

    def minimum(self) -> int:
        while self.del_mn[self.mn[0]]:
            self.del_mn[self.mn[0]] -= 1; heapq.heappop(self.mn)
        return self.mn[0]

print(run_ops(StockPriceCounterLazy, ops_example, args_example))

[None, None, None, 5, 10, None, 5, None, 2]


**Độ phức tạp Cách 4:** như Cách 2 (`O(log m)` khấu hao). Lưu ý tinh tế: nếu một giá cũ trùng với một giá **hiện tại** của thời điểm khác, ta xóa "một bản bất kỳ" của giá đó — vẫn đúng vì các bản cùng giá không phân biệt được.

## 8. TRACE từng bước

Trace Cách 2 trên ví dụ đề:

| Lệnh | d | latest | mx (đỉnh) | mn (đỉnh) | Kết quả |
|---|---|---|---|---|---|
| update(1,10) | {1:10} | 1 | (−10,1) | (10,1) | — |
| update(2,5) | {1:10, 2:5} | 2 | (−10,1) | (5,2) | — |
| current | | | | | 5 |
| maximum | | | (−10,1) hợp lệ | | 10 |
| update(1,3) | {1:3, 2:5} | 2 | (−10,1) **lỗi thời** | (3,1) | — |
| maximum | | | pop (−10,1) → (−5,2) | | 5 |
| update(4,2) | {1:3, 2:5, 4:2} | 4 | | (2,4) | — |
| minimum | | | | (2,4) hợp lệ | 2 |

In [11]:
class StockPriceTrace(StockPriceHeap):
    def update(self, timestamp, price):
        super().update(timestamp, price)
        print(f"update({timestamp},{price}): d={self.d}, latest={self.latest}, "
              f"đỉnh mx={self.mx[0]}, đỉnh mn={self.mn[0]}")
    def maximum(self):
        while self.d[self.mx[0][1]] != -self.mx[0][0]:
            print(f"   maximum: bỏ {self.mx[0]} vì d[{self.mx[0][1]}]={self.d[self.mx[0][1]]}")
            heapq.heappop(self.mx)
        print(f"maximum() = {-self.mx[0][0]}"); return -self.mx[0][0]
    def minimum(self):
        while self.d[self.mn[0][1]] != self.mn[0][0]:
            print(f"   minimum: bỏ {self.mn[0]}"); heapq.heappop(self.mn)
        print(f"minimum() = {self.mn[0][0]}"); return self.mn[0][0]
    def current(self):
        print(f"current() = d[{self.latest}] = {self.d[self.latest]}"); return self.d[self.latest]

print(run_ops(StockPriceTrace, ops_example, args_example))

update(1,10): d={1: 10}, latest=1, đỉnh mx=(-10, 1), đỉnh mn=(10, 1)
update(2,5): d={1: 10, 2: 5}, latest=2, đỉnh mx=(-10, 1), đỉnh mn=(5, 2)
current() = d[2] = 5
maximum() = 10
update(1,3): d={1: 3, 2: 5}, latest=2, đỉnh mx=(-10, 1), đỉnh mn=(3, 1)
   maximum: bỏ (-10, 1) vì d[1]=3
maximum() = 5
update(4,2): d={1: 3, 2: 5, 4: 2}, latest=4, đỉnh mx=(-5, 2), đỉnh mn=(2, 4)
minimum() = 2
[None, None, None, 5, 10, None, 5, None, 2]


In [12]:
# Một chuỗi có nhiều lần sửa liên tiếp để thấy rác tích tụ rồi bị dọn
ops = ["S", "update", "update", "update", "update", "minimum", "maximum", "update", "minimum", "current"]
args = [[], [5, 1], [5, 2], [5, 3], [3, 7], [], [], [9, 4], [], []]
print(run_ops(StockPriceTrace, ops, args))

update(5,1): d={5: 1}, latest=5, đỉnh mx=(-1, 5), đỉnh mn=(1, 5)
update(5,2): d={5: 2}, latest=5, đỉnh mx=(-2, 5), đỉnh mn=(1, 5)
update(5,3): d={5: 3}, latest=5, đỉnh mx=(-3, 5), đỉnh mn=(1, 5)
update(3,7): d={5: 3, 3: 7}, latest=5, đỉnh mx=(-7, 3), đỉnh mn=(1, 5)
   minimum: bỏ (1, 5)
   minimum: bỏ (2, 5)
minimum() = 3
maximum() = 7
update(9,4): d={5: 3, 3: 7, 9: 4}, latest=9, đỉnh mx=(-7, 3), đỉnh mn=(3, 5)
minimum() = 3
current() = d[9] = 4
[None, None, None, None, None, 3, 7, None, 3, 4]


## 9. Code nộp LeetCode

In [13]:
class StockPrice:

    def __init__(self):
        self.price_at = {}            # timestamp -> giá hiện tại
        self.latest = 0               # timestamp lớn nhất
        self.max_heap = []            # (-giá, timestamp), có thể chứa bản lỗi thời
        self.min_heap = []            # (giá, timestamp)

    def update(self, timestamp: int, price: int) -> None:
        self.price_at[timestamp] = price
        if timestamp > self.latest:
            self.latest = timestamp
        heapq.heappush(self.max_heap, (-price, timestamp))
        heapq.heappush(self.min_heap, (price, timestamp))

    def current(self) -> int:
        return self.price_at[self.latest]

    def maximum(self) -> int:
        h = self.max_heap
        while self.price_at[h[0][1]] != -h[0][0]:    # xóa lười bản ghi đã bị sửa
            heapq.heappop(h)
        return -h[0][0]

    def minimum(self) -> int:
        h = self.min_heap
        while self.price_at[h[0][1]] != h[0][0]:
            heapq.heappop(h)
        return h[0][0]

assert run_ops(StockPrice, ops_example, args_example) == expected_example
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Bất biến (Cách 2).** Tại mọi thời điểm:
1. `price_at` chứa đúng giá hiện tại của mỗi timestamp (ghi đè khi sửa).
2. Với mỗi timestamp `ts`, cặp `(price_at[ts], ts)` **có mặt** trong `min_heap` (và `(−price_at[ts], ts)` trong `max_heap`) — vì được đẩy vào ở lần `update` gần nhất của `ts` và chỉ bị pop khi lỗi thời.
3. Một phần tử `(p, ts)` trong heap là **hợp lệ** ⇔ `price_at[ts] == p`.

**Đúng của `minimum`.** Vòng `while` loại các phần tử lỗi thời ở đỉnh. Khi dừng, đỉnh `(p, ts)` hợp lệ và là nhỏ nhất trong heap, mà mọi giá hiện tại đều có mặt trong heap (bất biến 2) ⇒ `p` = min các giá hiện tại. Việc pop không phá bất biến 2 vì chỉ pop phần tử lỗi thời. (Nếu cùng `ts` có hai cặp `(p, ts)` giống hệt nhau — do update cùng giá hai lần — pop một cặp vẫn còn cặp kia, và cả hai đều "hợp lệ" nên không bị pop.)

**`current`.** `latest` là max của mọi timestamp đã thấy ⇒ `price_at[latest]` đúng định nghĩa.

## 11. Phân tích Time / Space

Gọi `m` = số lần `update`, `k` = số timestamp khác nhau, `q` = tổng số lời gọi.

| Cách | `update` | `current` | `max/min` | Space |
|---|---|---|---|---|
| 1. Brute | `O(1)` | `O(1)` | `O(k)` | `O(k)` |
| 2. Hai heap lười | `O(log m)` | `O(1)` | `O(log m)` khấu hao | `O(m)` |
| 3. SortedList | `O(log k)` (thật) / `O(k)` (Mini) | `O(1)` | `O(1)` | `O(k)` |
| 4. Heap + Counter | `O(log m)` | `O(1)` | `O(log m)` khấu hao | `O(m)` |

Đo trên `10⁵` lệnh ngẫu nhiên (nhiều lần sửa):

In [14]:
rng = random.Random(2034)
ops, args = ["S"], [[]]
for _ in range(10**5):
    r = rng.random()
    if r < 0.5 or len(ops) == 1:
        ops.append("update"); args.append([rng.randint(1, 30000), rng.randint(1, 10**9)])
    else:
        ops.append(rng.choice(["current", "maximum", "minimum"])); args.append([])
results = {}
for cls in (StockPriceHeap, StockPriceSorted, StockPriceCounterLazy, StockPrice):
    t = time.perf_counter(); results[cls.__name__] = run_ops(cls, ops, args)
    print(f"{cls.__name__:22s}: {time.perf_counter() - t:.3f}s")
assert len({tuple(v) for v in results.values()}) == 1
print("Các cài đặt nhanh cho cùng kết quả trên 10^5 lệnh")

StockPriceHeap        : 0.077s


StockPriceSorted      : 0.133s
StockPriceCounterLazy : 0.046s


StockPrice            : 0.057s
Các cài đặt nhanh cho cùng kết quả trên 10^5 lệnh


## 12. Bộ test

- Ví dụ đề.
- Biên: một `update` rồi hỏi cả ba; sửa cùng `ts` nhiều lần; `update` thời điểm **nhỏ hơn** `latest` (không đổi `current`); nhiều `ts` cùng giá rồi sửa một cái; sửa về đúng giá cũ.
- Đối chiếu mọi cài đặt với brute force trên chuỗi lệnh ngẫu nhiên (ít timestamp để có nhiều lần sửa; seed cố định).

In [15]:
classes = [StockPriceBrute, StockPriceHeap, StockPriceSorted, StockPriceCounterLazy, StockPrice]
def run_all(ops, args):
    outs = [run_ops(c, ops, args) for c in classes]
    for o in outs[1:]:
        assert o == outs[0], (ops, args, outs[0], o)
    return outs[0]

assert run_all(ops_example, args_example) == expected_example
assert run_all(["S", "update", "current", "maximum", "minimum"], [[], [7, 4], [], [], []]) == [None, None, 4, 4, 4]
assert run_all(["S", "update", "update", "update", "maximum", "minimum"], [[], [1, 9], [1, 2], [1, 6], [], []])[-2:] == [6, 6]
assert run_all(["S", "update", "update", "current"], [[], [10, 1], [3, 99], []])[-1] == 1          # ts nhỏ hơn latest
assert run_all(["S", "update", "update", "update", "maximum"], [[], [1, 5], [2, 5], [1, 1], []])[-1] == 5   # cùng giá
assert run_all(["S", "update", "update", "minimum"], [[], [1, 5], [1, 5], []])[-1] == 5             # sửa về đúng giá cũ
print("Ví dụ + biên: OK")

Ví dụ + biên: OK


In [16]:
rng = random.Random(42)
for _ in range(300):
    ops, args = ["S"], [[]]
    T = rng.choice([1, 3, 6, 20]); P = rng.choice([3, 10, 10**9])
    ops.append("update"); args.append([rng.randint(1, T), rng.randint(1, P)])
    for _ in range(rng.randint(1, 40)):
        if rng.random() < 0.5:
            ops.append("update"); args.append([rng.randint(1, T), rng.randint(1, P)])
        else:
            ops.append(rng.choice(["current", "maximum", "minimum"])); args.append([])
    run_all(ops, args)
print("300 chuỗi lệnh ngẫu nhiên: mọi cài đặt khớp brute force")

300 chuỗi lệnh ngẫu nhiên: mọi cài đặt khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Không xử lý việc **sửa** giá (giá cũ vẫn được tính vào max/min)

In [17]:
class WrongNoCorrection:
    def __init__(self):
        self.d = {}; self.latest = 0; self.mx = []; self.mn = []
    def update(self, timestamp, price):
        self.d[timestamp] = price; self.latest = max(self.latest, timestamp)
        heapq.heappush(self.mx, -price); heapq.heappush(self.mn, price)
    def current(self): return self.d[self.latest]
    def maximum(self): return -self.mx[0]               # SAI: không loại giá đã bị sửa
    def minimum(self): return self.mn[0]

print("sai:", run_ops(WrongNoCorrection, ops_example, args_example))
print("đúng:", expected_example)

sai: [None, None, None, 5, 10, None, 10, None, 2]
đúng: [None, None, None, 5, 10, None, 5, None, 2]


In [18]:
print("sửa (xóa lười):", run_ops(StockPrice, ops_example, args_example))

sửa (xóa lười): [None, None, None, 5, 10, None, 5, None, 2]


### Lỗi 2 — `current` trả giá của lần `update` **gần nhất** thay vì thời điểm **lớn nhất**

Bản ghi đến không theo thứ tự: một `update` với `ts` nhỏ hơn không làm thay đổi "giá mới nhất".

In [19]:
class WrongCurrent(StockPrice):
    def update(self, timestamp, price):
        super().update(timestamp, price)
        self.last_call_ts = timestamp
    def current(self):
        return self.price_at[self.last_call_ts]          # SAI

ops2, args2 = ["S", "update", "update", "current"], [[], [10, 1], [3, 99], []]
print("sai:", run_ops(WrongCurrent, ops2, args2)[-1], "| đúng:", run_ops(StockPrice, ops2, args2)[-1])

sai: 99 | đúng: 1


In [20]:
print("sửa (latest = max timestamp):", run_ops(StockPriceBrute, ops2, args2)[-1])

sửa (latest = max timestamp): 1


### Lỗi 3 — Kiểm "lỗi thời" chỉ bằng `ts in d` (không so giá)

Timestamp vẫn tồn tại sau khi sửa, nên kiểm `ts in d` không phát hiện được bản ghi cũ.

In [21]:
class WrongLazyCheck(StockPrice):
    def maximum(self):
        h = self.max_heap
        while h[0][1] not in self.price_at:               # SAI: phải so price_at[ts] với giá
            heapq.heappop(h)
        return -h[0][0]

print("sai:", run_ops(WrongLazyCheck, ops_example, args_example)[6], "| đúng:", expected_example[6])

sai: 10 | đúng: 5


In [22]:
print("sửa:", run_ops(StockPrice, ops_example, args_example)[6])

sửa: 5


### Lỗi 4 — SortedList: thêm giá mới nhưng quên xóa giá cũ (hoặc xóa **mọi** bản cùng giá)

Quên xóa thì giống Lỗi 1. Còn xóa mọi bản cùng giá (ví dụ dùng `set` thay vì danh sách có lặp) sẽ làm mất giá của **timestamp khác** có cùng giá.

In [23]:
class WrongSetPrices:
    def __init__(self):
        self.d = {}; self.latest = 0; self.prices = set()
    def update(self, timestamp, price):
        if timestamp in self.d:
            self.prices.discard(self.d[timestamp])         # SAI: set không đếm số bản
        self.d[timestamp] = price; self.prices.add(price); self.latest = max(self.latest, timestamp)
    def current(self): return self.d[self.latest]
    def maximum(self): return max(self.prices)
    def minimum(self): return min(self.prices)

ops3, args3 = ["S", "update", "update", "update", "maximum"], [[], [1, 5], [2, 5], [1, 1], []]
print("sai:", run_ops(WrongSetPrices, ops3, args3)[-1], "| đúng:", run_ops(StockPrice, ops3, args3)[-1])

sai: 1 | đúng: 5


In [24]:
print("sửa (danh sách có lặp):", run_ops(StockPriceSorted, ops3, args3)[-1])

sửa (danh sách có lặp): 5


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Bài thiết kế nhiều thao tác** ⇒ lập bảng "thao tác → cần biết gì → cấu trúc nào", rồi đồng bộ các cấu trúc với một **nguồn sự thật** (`dict`).
- **Max/min với cập nhật/xóa phần tử bất kỳ** ⇒ heap + **xóa lười**, hoặc **tập có thứ tự**.
- Xóa lười cần cách kiểm "còn hợp lệ không" **chính xác**: so với nguồn sự thật (`d[ts] == price`).
- "Mới nhất" theo **giá trị timestamp**, không theo **thứ tự lời gọi**.
- Test bằng **bộ chạy lệnh + brute force** trên chuỗi lệnh ngẫu nhiên — rất hiệu quả cho bài design.

## 15. Biến thể của bài

1. **LeetCode 295 — Find Median from Data Stream (Hard):** hai heap cân bằng.
2. **LeetCode 480 — Sliding Window Median (Hard):** hai heap + xóa lười.
3. **LeetCode 2353 — Design a Food Rating System:** heap theo nhóm + xóa lười (cùng kỹ thuật kiểm hợp lệ bằng `dict`).
4. **Thêm thao tác `delete(timestamp)`** — với xóa lười chỉ cần `del d[ts]` và kiểm `ts in d and d[ts] == price`.
5. **Thêm `median()`** (bài tập 5) — dùng danh sách có thứ tự.

Minh họa biến thể 4 (tự dựng, kiểm bằng brute force):

In [25]:
class StockPriceWithDelete(StockPrice):
    def delete(self, timestamp):
        del self.price_at[timestamp]
        if timestamp == self.latest:
            self.latest = max(self.price_at) if self.price_at else 0
    def maximum(self):
        h = self.max_heap
        while h[0][1] not in self.price_at or self.price_at[h[0][1]] != -h[0][0]:
            heapq.heappop(h)
        return -h[0][0]
    def minimum(self):
        h = self.min_heap
        while h[0][1] not in self.price_at or self.price_at[h[0][1]] != h[0][0]:
            heapq.heappop(h)
        return h[0][0]

class BruteWithDelete(StockPriceBrute):
    def delete(self, timestamp):
        del self.d[timestamp]
        self.latest = max(self.d) if self.d else 0

rng = random.Random(4)
for _ in range(300):
    a, b = StockPriceWithDelete(), BruteWithDelete()
    for _ in range(rng.randint(1, 30)):
        r = rng.random()
        if r < 0.5 or not b.d:
            ts, p = rng.randint(1, 5), rng.randint(1, 9); a.update(ts, p); b.update(ts, p)
        elif r < 0.65:
            ts = rng.choice(list(b.d)); a.delete(ts); b.delete(ts)
        else:
            if b.d:
                assert (a.current(), a.maximum(), a.minimum()) == (b.current(), b.maximum(), b.minimum())
print("Biến thể có delete khớp brute force")

Biến thể có delete khớp brute force


## 16. So sánh các cách

| Cách | Ý tưởng | `max/min` | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | `max(d.values())` | `O(k)` | `O(k)` | kiểm tra |
| 2. Hai heap lười | kiểm `d[ts] == price` ở đỉnh | `O(log m)` khấu hao | `O(m)` | **nộp bài**, không cần thư viện |
| 3. SortedList | xóa giá cũ, thêm giá mới | `O(1)` | `O(k)` | khi có `sortedcontainers` |
| 4. Heap + Counter | đếm số bản cần xóa theo giá | `O(log m)` khấu hao | `O(m)` | heap chỉ chứa giá |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `update(5,1)`, `update(5,2)`, `update(5,3)`, `update(3,7)`, rồi `minimum`, `maximum`, `current`.
2. **Tính tay:** vẽ nội dung `max_heap` (kể cả rác) sau chuỗi trên, và chỉ ra phần tử nào bị pop khi gọi `maximum`.
3. **Điền TODO:** `clean_top(heap, price_at, sign)` — hàm dùng chung cho cả max và min để loại phần tử lỗi thời ở đỉnh.
4. **Tự viết lại:** viết lại `StockPrice` (Cách 2) không nhìn code.
5. **Mini challenge:** thêm `median()` trả về giá trung vị "dưới" (`sorted(values)[(k−1)//2]`). Gợi ý: dùng `MiniSortedList` như Cách 3. Kiểm bằng brute force.

In [26]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def clean_top(heap, price_at, sign):
    # TODO: sign = 1 cho min-heap (price, ts), -1 cho max-heap (-price, ts)
    pass

In [27]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
class MyStockPrice:
    def __init__(self):
        pass  # TODO
    def update(self, timestamp, price):
        pass  # TODO
    def current(self):
        pass  # TODO
    def maximum(self):
        pass  # TODO
    def minimum(self):
        pass  # TODO

In [28]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
class StockPriceMedian:
    def __init__(self):
        pass  # TODO
    def update(self, timestamp, price):
        pass  # TODO
    def median(self):
        pass  # TODO

## 18. Gợi ý 3 mức

**Bài 1.** (1) Ba lần sửa `ts = 5`: chỉ giá 3 còn hiệu lực. (2) Giá hiện tại: `{5: 3, 3: 7}`. (3) `minimum = 3`, `maximum = 7`, `current = d[5] = 3`.

**Bài 2.** (1) `max_heap` có `(−1,5), (−2,5), (−3,5), (−7,3)`. (2) Đỉnh `(−7,3)` hợp lệ ⇒ không pop gì khi gọi `maximum`. (3) Với `minimum`: đỉnh `(1,5)` lỗi thời ⇒ pop, `(2,5)` lỗi thời ⇒ pop, `(3,5)` hợp lệ.

**Bài 3.** (1) Phần tử đỉnh `(v, ts)`. (2) Giá thật là `sign * v`. (3) Pop khi `price_at[ts] != sign * v`.

**Bài 4.** (1) `dict`, `latest`, hai heap. (2) `update` đẩy vào cả hai heap. (3) `maximum/minimum` dọn đỉnh rồi trả về.

**Bài 5.** (1) Giữ `MiniSortedList` các giá hiện tại. (2) Sửa: `remove(old)` rồi `add(new)`. (3) `median = sl[(len(sl) − 1) // 2]`.

## 19. Đáp án tham khảo

In [29]:
# Bài 1, 2
ops1 = ["S", "update", "update", "update", "update", "minimum", "maximum", "current"]
args1 = [[], [5, 1], [5, 2], [5, 3], [3, 7], [], [], []]
assert run_all(ops1, args1)[-3:] == [3, 7, 3]
sp = StockPrice()
for a in args1[1:5]: sp.update(*a)
print("Bài 1: [3, 7, 3] | Bài 2 — max_heap trước khi hỏi:", sorted(sp.max_heap), "| min_heap:", sorted(sp.min_heap))

Bài 1: [3, 7, 3] | Bài 2 — max_heap trước khi hỏi: [(-7, 3), (-3, 5), (-2, 5), (-1, 5)] | min_heap: [(1, 5), (2, 5), (3, 5), (7, 3)]


In [30]:
# Bài 3, 4
def clean_top_ans(heap, price_at, sign):
    while price_at[heap[0][1]] != sign * heap[0][0]:
        heapq.heappop(heap)
    return sign * heap[0][0]

class MyStockPriceAns:
    def __init__(self):
        self.d = {}; self.latest = 0; self.mx = []; self.mn = []
    def update(self, timestamp, price):
        self.d[timestamp] = price; self.latest = max(self.latest, timestamp)
        heapq.heappush(self.mx, (-price, timestamp)); heapq.heappush(self.mn, (price, timestamp))
    def current(self): return self.d[self.latest]
    def maximum(self): return clean_top_ans(self.mx, self.d, -1)
    def minimum(self): return clean_top_ans(self.mn, self.d, 1)

classes.append(MyStockPriceAns)
rng = random.Random(3)
for _ in range(200):
    ops, args = ["S", "update"], [[], [rng.randint(1, 4), rng.randint(1, 9)]]
    for _ in range(rng.randint(1, 30)):
        if rng.random() < 0.5: ops.append("update"); args.append([rng.randint(1, 4), rng.randint(1, 9)])
        else: ops.append(rng.choice(["current", "maximum", "minimum"])); args.append([])
    run_all(ops, args)
classes.pop()
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [31]:
# Bài 5
class StockPriceMedianAns:
    def __init__(self):
        self.d = {}; self.sl = MiniSortedList()
    def update(self, timestamp, price):
        if timestamp in self.d: self.sl.remove(self.d[timestamp])
        self.d[timestamp] = price; self.sl.add(price)
    def median(self):
        return self.sl[(len(self.sl) - 1) // 2]

rng = random.Random(5)
for _ in range(300):
    a = StockPriceMedianAns(); d = {}
    for _ in range(rng.randint(1, 25)):
        ts, p = rng.randint(1, 6), rng.randint(1, 20)
        a.update(ts, p); d[ts] = p
        vals = sorted(d.values())
        assert a.median() == vals[(len(vals) - 1) // 2]
print("Bài 5: OK")

Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "heap + xóa lười theo nguồn sự thật":**

```python
truth[key] = value
heapq.heappush(h, (value, key))          # không xóa bản cũ
...
while truth[h[0][1]] != h[0][0]:         # đỉnh lỗi thời?
    heapq.heappop(h)
return h[0][0]
```

In [32]:
def template_min(h, truth):
    while truth[h[0][1]] != h[0][0]:
        heapq.heappop(h)
    return h[0][0]
truth, h = {}, []
for ts, p in [(1, 10), (2, 5), (1, 3), (4, 2)]:
    truth[ts] = p; heapq.heappush(h, (p, ts))
print("min =", template_min(h, truth))

min = 2


**Checklist:**

- [ ] `update` ghi đè `dict` (sửa giá) và cập nhật `latest = max(latest, ts)`.
- [ ] `current` dùng `latest` (timestamp lớn nhất), không dùng lời gọi gần nhất.
- [ ] Max/min không được tính giá đã bị sửa: xóa lười (so `d[ts] == price`) hoặc xóa đúng một bản trong SortedList.
- [ ] Không dùng `set` cho các giá (giá có thể trùng giữa các timestamp).
- [ ] Max-heap bằng đổi dấu.
- [ ] Test bằng bộ chạy lệnh + brute force với ít timestamp (nhiều lần sửa).